In [6]:
import pandas as pd
from data_preprocessing import  separate_tf_genes, prepare_for_inference
from tf_utils import map_tf_ids
import db as db
from utils_network import *
from inference import *

In [7]:
config_file = "/data_nfs/og86asub/alternet-project/alternet/configs/MAGNet_NF.yaml"
import yaml

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)


# Load necessary data and annotations

In [8]:
biomart_path = '/data_nfs/og86asub/alternet-project/alternet/data/biomart.txt'
tf_list_path = '/data_nfs/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'

biomart = pd.read_csv(biomart_path, sep='\t')
tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)

# create TF Database
tf_database = db.create_transcipt_annotation_database(tf_list=tf_list, appris_path= config['appris'], digger_path=config['digger'])


/data_nfs/og86asub/alternet-project/alternet/src/alternet/db.py:32: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  digger = pd.read_csv(digger_path)


In [9]:
## LOAD gene and transcript data
transcript_data = pd.read_csv(config['transcript_data'], index_col=0)

## GET the gene ids associated with transcripts
gene_ids = transcript_data['gene_id'].unique()




In [13]:
transcript_data

,transcript_id,gene_id,C00039,C00055,C00074,C00085,C00105,C00120,C00132,C00158,...,P01626,P01627,P01628,P01629,P01630,P01631,P01634,P01635,P01639,P01640
0,ENST00000473530,ENSG00000244682,2.276150,2.040930,1.564230,2.629880,3.224940,0.081621,3.577190,3.275500,...,2.249030,0.754787,0.000000,1.594620,0.000000,0.031552,1.032080,2.658050,1.475990,0.012955
1,ENST00000467903,ENSG00000244682,2.167410,2.788660,3.829120,3.177140,4.211690,0.176553,6.959880,3.485350,...,4.777670,2.296700,0.256739,7.109500,0.226315,0.453683,1.436180,2.298990,3.807010,0.155549
2,ENST00000508651,ENSG00000244682,1.950990,2.569730,3.989870,3.134920,5.018950,0.110165,4.027630,1.973540,...,4.473280,1.117980,0.185469,3.784620,0.146539,0.085379,1.271050,1.629250,3.130670,0.000000
3,ENST00000626647,ENSG00000280938,0.049179,0.019637,0.017334,0.489663,0.000000,0.105393,0.517216,0.029199,...,0.494979,3.469800,6.911700,1.803350,1.116030,0.850385,0.507154,0.000000,1.293200,1.458430
4,ENST00000633377,ENSG00000282181,3.231750,1.134710,3.935840,0.548367,0.007815,0.319113,3.020600,1.936820,...,2.497850,0.987592,0.007706,0.507429,0.257754,0.544622,0.007459,1.090220,0.713997,1.031000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39310,ENST00000569097,ENSG00000197081,0.821758,0.377072,0.413925,0.907076,0.476571,0.098923,0.395161,0.379031,...,0.957061,0.418827,0.047140,0.320178,0.066159,0.132223,0.285653,0.631274,0.421508,0.062954
39311,ENST00000602699,ENSG00000225366,0.050290,0.014330,0.035171,0.041823,0.037060,0.000000,0.065104,0.016308,...,0.044029,0.000000,0.000000,0.025864,0.024941,0.048181,0.038492,0.051400,0.017281,0.014494
39312,ENST00000594408,ENSG00000182986,2.056060,1.068780,0.909087,1.211280,1.235250,0.022664,1.194140,1.571050,...,1.833670,0.982506,0.017389,0.592679,0.000000,0.043223,0.683917,1.783930,1.038960,0.000000
39313,ENST00000510091,ENSG00000205482,0.397195,0.139399,0.137387,0.238294,0.456423,0.055239,0.127514,0.330749,...,1.001040,0.285062,0.049527,0.056683,0.000000,0.000000,0.040746,0.233459,0.148919,0.000000


In [16]:
gene_data = transcript_data.groupby('gene_id').sum()

In [18]:
gene_data = gene_data.drop(columns={'transcript_id'})

In [19]:
gene_data.index.name = 'gene_id'
gene_data = gene_data.reset_index()
gene_data = gene_data.iloc[0:40, ]

In [20]:

## Subset the samples of interest
sample_attributes = pd.read_csv(config['sample_attributes'])
sample_attributes = sample_attributes.loc[:, ['sample_name', 'etiology']]
samples = sample_attributes[sample_attributes['etiology'] == config['tissue']]
samples = samples['sample_name'].tolist()


## Get unified gene and transcript table
gene_data = gene_data.loc[:, ['gene_id'] + samples ]
transcript_data = transcript_data.loc[:,['gene_id', 'transcript_id'] + samples]


In [88]:
transcript_data = transcript_data[transcript_data.gene_id.isin(gene_data.gene_id)]

In [89]:
tf_iso_counts, _ = separate_tf_genes(data=transcript_data, tf_list=tf_list, data_column='transcript_id', biomart_column='Transcript stable ID')

In [90]:
tf_gene_counts, target_gene_counts = separate_tf_genes(data=gene_data, tf_list=tf_list, data_column='gene_id', biomart_column='Gene stable ID')

In [91]:
data_canonical, data_asware, target_gene_list = prepare_for_inference(tf_iso_counts, tf_gene_counts, target_gene_counts, transcript_column='transcript_id', gene_column='gene_id')

Preparing data for inference: Separation into canonical and as-aware gene expression data


In [92]:
tf_iso_counts


,gene_id,transcript_id,C00039,C00055,C00074,C00085,C00105,C00120,C00132,C00158,...,P01600,P01603,P01610,P01616,P01621,P01622,P01623,P01626,P01628,P01635
5909,ENSG00000002330,ENST00000394532,0.605,0.129,0.922,0.101,0.178,1.171,1.080,0.604,...,0.000,1.863,4.104,0.110,0.846,2.648,1.002,0.118,1.059,0.255
5910,ENSG00000002330,ENST00000309032,4.650,10.543,14.435,4.612,11.971,32.966,8.241,8.765,...,4.035,22.083,48.003,1.338,61.267,36.377,10.078,9.418,60.931,3.175
5911,ENSG00000002330,ENST00000492141,0.759,0.215,0.390,0.487,0.308,1.545,0.647,0.223,...,0.282,0.155,0.564,0.159,0.527,0.201,0.573,1.278,0.572,0.711
5912,ENSG00000002330,ENST00000544271,0.480,1.078,1.049,0.334,1.103,0.880,0.949,1.441,...,0.443,4.532,9.830,0.269,2.731,7.120,0.503,2.001,1.754,0.383
22889,ENSG00000001497,ENST00000374807,2.467,1.093,4.324,1.140,1.902,0.000,2.691,1.939,...,2.186,2.185,1.818,1.107,0.000,1.410,2.558,2.237,0.000,1.172
22890,ENSG00000001497,ENST00000374811,3.397,3.568,3.103,3.469,3.072,1.991,3.612,3.724,...,4.344,2.018,3.624,1.677,1.978,2.376,3.360,3.261,1.341,2.633
22891,ENSG00000001497,ENST00000484069,3.668,0.873,1.950,1.908,1.228,0.336,1.388,1.339,...,1.354,0.992,1.015,1.588,0.339,0.488,1.656,2.740,0.341,1.767
22892,ENSG00000001497,ENST00000469091,6.124,2.061,1.994,2.998,1.911,4.647,2.508,2.994,...,2.381,1.412,2.423,2.696,4.847,0.856,3.627,4.091,4.766,3.177
27939,ENSG00000001167,ENST00000341376,7.683,4.104,4.853,4.318,5.405,0.897,3.440,4.189,...,4.121,3.055,4.433,4.572,0.866,3.646,2.542,4.437,0.818,6.868


In [94]:
# get isoform categories
isoform_categories = isoform_categorization(tf_iso_counts, tf_gene_counts, threshold_dominance=80)


            gene_id  C00039  C00055  C00074  C00085  C00105  C00120  C00132  \
9   ENSG00000001167   7.683   4.104   4.853   4.318   5.405   0.897   3.440   
12  ENSG00000001497  15.656   7.595  11.371   9.515   8.113   6.974  10.199   
19  ENSG00000002330   6.495  11.966  16.796   5.534  13.560  36.562  10.917   

    C00158  C00168  ...  P01603  P01610  P01616  P01621  P01622  P01623  \
9    4.189   2.409  ...   3.055   4.433   4.572   0.866   3.646   2.542   
12   9.996  12.917  ...   6.606   8.879   7.068   7.164   5.130  11.201   
19  11.033  18.672  ...  28.633  62.502   1.875  65.370  46.346  12.157   

    P01626  P01628  P01635  sum_gene_expression  
9    4.437   0.818   6.868              598.264  
12  12.328   6.447   8.749             1639.394  
19  12.816  64.316   4.525             4108.186  

[3 rows x 168 columns]


In [95]:
tf_gene_counts

,gene_id,C00039,C00055,C00074,C00085,C00105,C00120,C00132,C00158,C00168,...,P01600,P01603,P01610,P01616,P01621,P01622,P01623,P01626,P01628,P01635
9,ENSG00000001167,7.683,4.104,4.853,4.318,5.405,0.897,3.440,4.189,2.409,...,4.121,3.055,4.433,4.572,0.866,3.646,2.542,4.437,0.818,6.868
12,ENSG00000001497,15.656,7.595,11.371,9.515,8.113,6.974,10.199,9.996,12.917,...,10.265,6.606,8.879,7.068,7.164,5.130,11.201,12.328,6.447,8.749
19,ENSG00000002330,6.495,11.966,16.796,5.534,13.560,36.562,10.917,11.033,18.672,...,4.761,28.633,62.502,1.875,65.370,46.346,12.157,12.816,64.316,4.525


In [96]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
isoform_categories

,transcript_id,gene_id,sum_transcript_expression,sum_gene_expression,percentage,max_percentage,min_percentage,std_percentage,isoform_category
0,ENST00000394532,ENSG00000002330,168.174,4108.186,4.094,86.225,2.390,40.867,non-dominant
1,ENST00000309032,ENSG00000002330,3542.295,4108.186,86.225,86.225,2.390,40.867,dominant
2,ENST00000492141,ENSG00000002330,98.202,4108.186,2.390,86.225,2.390,40.867,non-dominant
3,ENST00000544271,ENSG00000002330,299.516,4108.186,7.291,86.225,2.390,40.867,non-dominant
4,ENST00000374807,ENSG00000001497,309.502,1639.394,18.879,34.838,12.950,10.784,balanced
5,ENST00000374811,ENSG00000001497,546.445,1639.394,33.332,34.838,12.950,10.784,balanced
6,ENST00000484069,ENSG00000001497,212.309,1639.394,12.950,34.838,12.950,10.784,balanced
7,ENST00000469091,ENSG00000001497,571.137,1639.394,34.838,34.838,12.950,10.784,balanced
8,ENST00000341376,ENSG00000001167,598.264,598.264,100.000,100.000,100.000,NaN,dominant


In [97]:
as_aware_grn, canonical_grn = inference(config, 2, tf_list=tf_list, data_canonical=data_canonical, data_asware=data_asware, target_gene_list=target_gene_list, aggregate=True)


Starting inference ...


/data_nfs/og86asub/alternet-project/alternet/.pixi/envs/default/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38629 instead
  warnings.warn(


Computing network
Computing network
Computing network
Computing network
Inference complete
Aggregate results
Going through edges of each df ...


100%|██████████| 333/333 [00:00<00:00, 654500.11it/s]


Going through edges of each df ...


100%|██████████| 111/111 [00:00<00:00, 1531472.84it/s]


In [99]:
canonical_grn

,source,target,frequency,mean_importance,median_importance
0,ENSG00000001167,ENSG00000001629,2,169.258,171.127
1,ENSG00000001167,ENSG00000003056,2,171.919,186.820
2,ENSG00000001167,ENSG00000003393,2,150.924,152.407
3,ENSG00000001167,ENSG00000001561,2,133.648,134.837
4,ENSG00000001167,ENSG00000001460,2,114.212,121.778
...,...,...,...,...,...
106,ENSG00000002330,ENSG00000002549,2,18.644,26.240
107,ENSG00000002330,ENSG00000000460,2,10.091,10.950
108,ENSG00000001497,ENSG00000000460,2,9.237,9.560
109,ENSG00000001497,ENSG00000000005,2,7.867,9.063


In [104]:
#filter aggregate
as_aware_grn = filter_aggregated(as_aware_grn, threshold_frequency=1, threshold_importance=0.3)
canonical_grn = filter_aggregated(canonical_grn, threshold_frequency=1, threshold_importance=0.3)


In [141]:

net_AS = add_edge_key(as_aware_grn, biomart, source_column = 'source')
net_canonical = add_edge_key(canonical_grn, biomart, type='canonical', source_column='source')


ds
ds


In [151]:

# Categorize edges into gene-unique, isoform-unique, common
common_edges = get_common_edges(net_canonical, net_AS)
gene_unique, isoform_unique = get_diff(net_canonical, net_AS)

print('Number of edges in each category')
print('Number of edges in gene-exclusive: ', len(gene_unique))
print('Number of edges in isoform-exclusive: ', len(isoform_unique))
print('Number of edges in common interactions: ', len(common_edges))


Number of edges in each category
Number of edges in gene-exclusive:  1
Number of edges in isoform-exclusive:  19
Number of edges in common interactions:  20


In [157]:
common_edges


,source,target,frequency,mean_importance,median_importance,source_transcript,source_gene,source_transcript_name,source_gene_name,target_gene,target_gene_name,type,edge_key
0,ENST00000341376,ENSG00000001629,2,98.606,99.278,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001629,ANKIB1,AS,ENSG00000001167_ENSG00000001629
1,ENST00000341376,ENSG00000003756,2,89.960,96.038,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003756,RBM5,AS,ENSG00000001167_ENSG00000003756
2,ENST00000341376,ENSG00000003056,2,89.267,93.775,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003056,M6PR,AS,ENSG00000001167_ENSG00000003056
3,ENST00000341376,ENSG00000001561,2,74.263,86.639,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001561,ENPP4,AS,ENSG00000001167_ENSG00000001561
4,ENST00000341376,ENSG00000003393,2,77.451,85.535,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003393,ALS2,AS,ENSG00000001167_ENSG00000003393
5,ENST00000341376,ENSG00000001084,2,58.718,68.953,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001084,GCLC,AS,ENSG00000001167_ENSG00000001084
6,ENST00000544271,ENSG00000002822,2,71.203,74.353,ENST00000544271,ENSG00000002330,BAD-206,BAD,ENSG00000002822,MAD1L1,AS,ENSG00000002330_ENSG00000002822
7,ENST00000341376,ENSG00000003989,2,63.763,69.545,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003989,SLC7A2,AS,ENSG00000001167_ENSG00000003989
8,ENST00000341376,ENSG00000001460,2,55.771,57.909,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001460,STPG1,AS,ENSG00000001167_ENSG00000001460
9,ENST00000394532,ENSG00000002822,2,44.055,47.191,ENST00000394532,ENSG00000002330,BAD-203,BAD,ENSG00000002822,MAD1L1,AS,ENSG00000002330_ENSG00000002822


In [162]:
common_edges['source_descriptor'] = common_edges['source_gene']+':'+common_edges['source_transcript']
common_edges['target_descriptor'] = common_edges['target_gene']+':'+common_edges['target_gene']

In [165]:
common_edges

,source,target,frequency,mean_importance,median_importance,source_transcript,source_gene,source_transcript_name,source_gene_name,target_gene,target_gene_name,type,edge_key,source_descriptor,target_descriptor
0,ENST00000341376,ENSG00000001629,2,98.606,99.278,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001629,ANKIB1,AS,ENSG00000001167_ENSG00000001629,ENSG00000001167:ENST00000341376,ENSG00000001629:ENSG00000001629
1,ENST00000341376,ENSG00000003756,2,89.960,96.038,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003756,RBM5,AS,ENSG00000001167_ENSG00000003756,ENSG00000001167:ENST00000341376,ENSG00000003756:ENSG00000003756
2,ENST00000341376,ENSG00000003056,2,89.267,93.775,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003056,M6PR,AS,ENSG00000001167_ENSG00000003056,ENSG00000001167:ENST00000341376,ENSG00000003056:ENSG00000003056
3,ENST00000341376,ENSG00000001561,2,74.263,86.639,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001561,ENPP4,AS,ENSG00000001167_ENSG00000001561,ENSG00000001167:ENST00000341376,ENSG00000001561:ENSG00000001561
4,ENST00000341376,ENSG00000003393,2,77.451,85.535,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003393,ALS2,AS,ENSG00000001167_ENSG00000003393,ENSG00000001167:ENST00000341376,ENSG00000003393:ENSG00000003393
5,ENST00000341376,ENSG00000001084,2,58.718,68.953,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001084,GCLC,AS,ENSG00000001167_ENSG00000001084,ENSG00000001167:ENST00000341376,ENSG00000001084:ENSG00000001084
6,ENST00000544271,ENSG00000002822,2,71.203,74.353,ENST00000544271,ENSG00000002330,BAD-206,BAD,ENSG00000002822,MAD1L1,AS,ENSG00000002330_ENSG00000002822,ENSG00000002330:ENST00000544271,ENSG00000002822:ENSG00000002822
7,ENST00000341376,ENSG00000003989,2,63.763,69.545,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000003989,SLC7A2,AS,ENSG00000001167_ENSG00000003989,ENSG00000001167:ENST00000341376,ENSG00000003989:ENSG00000003989
8,ENST00000341376,ENSG00000001460,2,55.771,57.909,ENST00000341376,ENSG00000001167,NFYA-201,NFYA,ENSG00000001460,STPG1,AS,ENSG00000001167_ENSG00000001460,ENSG00000001167:ENST00000341376,ENSG00000001460:ENSG00000001460
9,ENST00000394532,ENSG00000002822,2,44.055,47.191,ENST00000394532,ENSG00000002330,BAD-203,BAD,ENSG00000002822,MAD1L1,AS,ENSG00000002330_ENSG00000002822,ENSG00000002330:ENST00000394532,ENSG00000002822:ENSG00000002822


In [163]:
target_genes = set(common_edges['target_gene']).difference(set(common_edges['source_gene']))

In [164]:
gene_transcript_map = defaultdict(set)

# Iterate through each pair of (gene, transcript) from the DataFrame using zip.
for gene, transcript in zip(common_edges['source_gene'], common_edges['source_transcript']):
    # Add the transcript to the set associated with the current gene.
    # The set automatically handles duplicate transcripts, ensuring each
    # transcript is listed only once for a given gene.
    if not pd.isna(transcript):
        gene_transcript_map[gene].add(transcript)


# Convert the sets of transcripts into sorted lists to match the desired output format.
# A dictionary comprehension is used for a clean and concise conversion.
node_data = {
    gene: sorted(list(transcripts))
    for gene, transcripts in gene_transcript_map.items()
}

# Add the genes without targets to the node_data dictionary with an empty list as their value.
for gene in target_genes:
    node_data[gene] = []



In [161]:
import graphviz

s = graphviz.Digraph(
    'gene_transcripts_expanded',
    filename='gene_transcripts_expanded.gv',
    graph_attr={
        'splines': 'spline',  # Added splines attribute
        'rankdir': 'neato'
    }
)
# Set global node and edge attributes for a consistent look.
s.attr('node', shape='record', style='rounded, filled', height='0.1')
s.attr('edge', color='grey', arrowhead='vee')

# Assume common_edges is your data source
# For this example, let's use a simple dictionary

for gene, transcripts in node_data.items():
    # Create a list of formatted transcript strings with ports
    transcript_ports = [f"<{t}>{t}" for t in transcripts]

    # Join the gene and the formatted transcripts for the node label
    label = f"{{<{gene}>{gene}|{'|'.join(transcript_ports)}}}"

    # Use the new formatted string to create the node
    s.node(gene, label)

for idx, row in common_edges.iterrows():

    # Add some edges to demonstrate the 'ortho' splines
    s.edge(row['source_descriptor'], row['target_descriptor'])

# Render the graph
s.render(view=True)

KeyError: 'source_descriptor'

In [153]:
plausibility_filtered = plausibility_filtering(config, isoform_unique, isoform_categories, tf_database)
